# Neutrónica con COBAYA
## Distribuciones de potencia en el reactor NuScale

**Máster THN — UPM | Práctica 1**

---

En este notebook analizamos las distribuciones de potencia del reactor NuScale calculadas con el código neutrónico **COBAYA**. Estudiamos:

1. Lectura de los archivos de salida COBAYA (`.CON` / `.txt`)
2. Distribución **radial** de potencia: núcleo completo y ensamblaje de combustible
3. Distribución **axial** de potencia: perfil por altura
4. Comparación de casos: REF, SIM y VAC
5. Análisis de **sensibilidad**: variación de potencia y temperatura
6. Factores de forma axial y factores de ingeniería


---
## 0. Configuración


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import pathlib
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS — Ajusta DATA_ROOT a tu entorno
# ============================================================
DATA_ROOT = pathlib.Path(
    r"C:\Users\pablo\OneDrive - Universidad Politécnica de Madrid\THN\Prácticas"
)
P1 = DATA_ROOT / "Practica 1"

# Archivos COBAYA disponibles
ARCHIVOS = {
    'REF_CORE': P1 / 'REF_CORE.txt',
    'SIM_CORE': P1 / 'SIM_CORE.txt',
    'VAC_CORE': P1 / 'VAC_CORE.txt',
    'REF_FA':   P1 / 'REF_FA.txt',
    'SIM_FA':   P1 / 'SIM_FA.txt',
    'VAC_FA':   P1 / 'VAC_FA.txt',
}

for nombre, ruta in ARCHIVOS.items():
    try:
        exists = ruta.exists()
    except Exception:
        exists = False
    status = "OK" if exists else "NO"
    print(f"  {status} {nombre}: {ruta.name}")

---
## 1. Lectura de archivos COBAYA

Los archivos `.txt` exportados por COBAYA contienen la distribución de potencia normalizada por ensamblaje/pin en formato tabular. La potencia está normalizada de manera que la media del núcleo es 1.0.

La estructura típica del archivo es:
```
# Distribución radial de potencia — caso REF
# Fila  Col  Potencia_relativa
   1     3     0.823
   1     4     0.945
   ...
```

Para la distribución axial, el formato incluye la cota `z` (cm) y la potencia relativa en esa posición.


In [ ]:
def leer_distribucion_cobaya(filepath):
    """
    Lee un archivo de distribución de potencia COBAYA.
    
    Formato esperado: columnas numéricas separadas por espacios,
    con líneas de comentario que empiezan por '#'.
    
    Returns
    -------
    pd.DataFrame con las columnas del archivo.
    """
    filepath = pathlib.Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {filepath}")
    
    # Detectar número de columnas ignorando comentarios
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        lineas = [l.strip() for l in f if l.strip() and not l.strip().startswith('#')]
    
    if not lineas:
        raise ValueError(f"Archivo vacío o sin datos: {filepath}")
    
    n_cols = len(lineas[0].split())
    
    df = pd.read_csv(
        filepath,
        sep=r'\s+',
        comment='#',
        header=None,
    )
    return df


def leer_distribucion_radial(filepath):
    """
    Lee la distribución radial de potencia normalizada.
    
    Asume formato: fila col potencia (3 columnas),
    o una matriz directamente con la disposición del núcleo.
    """
    df = leer_distribucion_cobaya(filepath)
    
    if df.shape[1] == 3:
        # Formato (fila, col, potencia)
        df.columns = ['fila', 'col', 'potencia']
        pivot = df.pivot(index='fila', columns='col', values='potencia')
        return pivot
    else:
        # Asumimos que ya es una matriz
        return df


def leer_distribucion_axial(filepath, col_z=0, col_p=1):
    """
    Lee la distribución axial de potencia.
    
    Parameters
    ----------
    col_z : índice de columna con la cota axial (cm)
    col_p : índice de columna con la potencia relativa
    """
    df = leer_distribucion_cobaya(filepath)
    return df.iloc[:, col_z].values, df.iloc[:, col_p].values


print("Funciones de lectura definidas.")
print("Nota: Si los archivos no están disponibles, las siguientes celdas mostrarán datos de ejemplo.")

---
## 2. Distribución radial de potencia

La distribución radial de potencia muestra cómo se distribuye la potencia entre los ensamblajes de combustible del núcleo. Un núcleo bien diseñado tiene una distribución lo más plana posible para evitar puntos calientes.

### 2.1 Posiciones del núcleo NuScale

El núcleo del NuScale tiene 37 ensamblajes en disposición aproximadamente circular (radio de 3 ensamblajes).


In [ ]:
# Mapa del núcleo NuScale: (fila, col) de cada ensamblaje
# Numeración estándar de COBAYA para el núcleo de 37 FA
CORE_MAP = [
    (0,2),(0,3),(0,4),
    (1,1),(1,2),(1,3),(1,4),(1,5),
    (2,0),(2,1),(2,2),(2,3),(2,4),(2,5),(2,6),
    (3,0),(3,1),(3,2),(3,3),(3,4),(3,5),(3,6),
    (4,0),(4,1),(4,2),(4,3),(4,4),(4,5),(4,6),
    (5,1),(5,2),(5,3),(5,4),(5,5),
    (6,2),(6,3),(6,4),
]

N_FA = len(CORE_MAP)
print(f"Número de ensamblajes en el núcleo: {N_FA}")

# Crear diccionario de posición -> índice
pos_to_idx = {pos: i for i, pos in enumerate(CORE_MAP)}

In [ ]:
def plot_distribucion_radial_nucleo(potencias, titulo='Distribución Radial de Potencia',
                                     core_map=CORE_MAP, ax=None, cmap='hot_r',
                                     annotate=True, vmin=None, vmax=None):
    """
    Dibuja la distribución radial de potencia en el mapa del núcleo.
    
    Parameters
    ----------
    potencias : array-like de longitud N_FA con las potencias relativas
    titulo    : título del gráfico
    annotate  : si True, escribe el valor en cada celda
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    else:
        fig = ax.figure
    
    potencias = np.asarray(potencias)
    vmin = vmin or potencias.min()
    vmax = vmax or potencias.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap)
    
    for i, (row, col) in enumerate(core_map):
        color = cmap_obj(norm(potencias[i]))
        rect = mpatches.FancyBboxPatch(
            (col * 1.05, (6 - row) * 1.05), 0.95, 0.95,
            boxstyle="round,pad=0.03",
            facecolor=color, edgecolor='gray', linewidth=0.8
        )
        ax.add_patch(rect)
        if annotate:
            ax.text(
                col * 1.05 + 0.475, (6 - row) * 1.05 + 0.475,
                f"{potencias[i]:.3f}",
                ha='center', va='center', fontsize=7.5,
                color='black' if potencias[i] < 0.7*vmax else 'white',
                fontweight='bold'
            )
    
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Potencia relativa', shrink=0.7)
    
    ax.set_xlim(-0.3, 7.6)
    ax.set_ylim(-0.3, 7.6)
    ax.set_aspect('equal')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.axis('off')
    
    return fig, ax


print("Función de visualización definida.")

In [ ]:
# Cargar distribuciones radiales del núcleo
# Si los archivos existen, se leen; si no, se usa un ejemplo sintético

def generar_datos_ejemplo_radial(n_fa=37, seed=42):
    """Genera una distribución radial sintética (parabólica + ruido)."""
    rng = np.random.default_rng(seed)
    core_map = CORE_MAP
    center = (3.0, 3.0)
    distancias = [np.sqrt((r-center[0])**2 + (c-center[1])**2) for r, c in core_map]
    max_d = max(distancias)
    potencias = np.array([1.4 - 0.5*(d/max_d)**2 for d in distancias])
    potencias += rng.normal(0, 0.02, n_fa)
    potencias = potencias / potencias.mean()  # Normalizar a media=1
    return potencias


datos_radiales = {}
for caso in ['REF', 'SIM', 'VAC']:
    archivo = ARCHIVOS.get(f'{caso}_CORE')
    try:
        df = leer_distribucion_cobaya(archivo)
        # Adaptar según formato real del archivo
        if df.shape[1] == 1:
            potencias = df.iloc[:, 0].values
        elif df.shape[1] == 3:
            # (fila, col, potencia)
            potencias = df.iloc[:, 2].values
        else:
            potencias = df.values.flatten()[:N_FA]
        datos_radiales[caso] = potencias[:N_FA]
        print(f"  ✓ {caso}: leído desde archivo")
    except Exception as e:
        datos_radiales[caso] = generar_datos_ejemplo_radial(seed={'REF':0,'SIM':1,'VAC':2}[caso])
        print(f"  ~ {caso}: usando datos de ejemplo ({e})")

In [ ]:
# Visualizar los tres casos
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Escala común para comparación justa
all_vals = np.concatenate(list(datos_radiales.values()))
vmin, vmax = all_vals.min(), all_vals.max()

for ax, (caso, potencias) in zip(axes, datos_radiales.items()):
    plot_distribucion_radial_nucleo(
        potencias, 
        titulo=f'Distribución Radial — {caso}',
        ax=ax, cmap='hot_r', vmin=vmin, vmax=vmax
    )

plt.suptitle('Comparación de distribuciones radiales: REF vs SIM vs VAC',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas de las distribuciones
print("=" * 55)
print(f"{'Caso':<8} {'Min':>8} {'Max':>8} {'Media':>8} {'σ':>8} {'F_xy':>8}")
print("-" * 55)
for caso, p in datos_radiales.items():
    Fxy = p.max() / p.mean()
    print(f"{caso:<8} {p.min():>8.4f} {p.max():>8.4f} {p.mean():>8.4f} {p.std():>8.4f} {Fxy:>8.4f}")
print("=" * 55)
print("\nF_xy = factor de pico radial (P_max / P_media)")

### 2.2 Distribución radial en el ensamblaje de combustible (pin-by-pin)

A mayor resolución, analizamos la distribución de potencia pin-a-pin dentro de un ensamblaje individual (17×17 varillas de combustible en el diseño Westinghouse).


In [ ]:
def generar_datos_FA_ejemplo(caso='REF', seed=0):
    """Genera distribución pin-by-pin sintética para un FA 17x17."""
    rng = np.random.default_rng(seed)
    N = 17
    x = np.linspace(-1, 1, N)
    X, Y = np.meshgrid(x, x)
    # Perfil parabólico con asimetría según caso
    if caso == 'REF':
        data = 1.2 - 0.5*(X**2 + Y**2)
    elif caso == 'SIM':
        data = 1.2 - 0.5*(X**2 + Y**2) - 0.2*np.exp(-(X-0.3)**2 - (Y-0.3)**2)
    else:  # VAC
        data = 1.0 - 0.3*(X**2 + Y**2) - 0.4*np.exp(-X**2 - Y**2)
    data += rng.normal(0, 0.03, (N, N))
    # Posiciones de barras de control (vacías)
    ctrl_pos = [(0,5),(0,8),(0,11),(5,0),(5,5),(5,8),(5,11),(5,16),
                (8,0),(8,5),(8,8),(8,11),(8,16),(11,0),(11,5),(11,8),(11,11),(11,16),
                (16,5),(16,8),(16,11)]
    for r, c in ctrl_pos:
        data[r, c] = 0.0  # Posición vacía
    data = np.maximum(data, 0)
    data = data / data[data>0].mean()
    return data


datos_FA = {}
for caso in ['REF', 'SIM', 'VAC']:
    archivo = ARCHIVOS.get(f'{caso}_FA')
    try:
        df = leer_distribucion_cobaya(archivo)
        data = df.values
        if data.shape == (17, 17):
            datos_FA[caso] = data
            print(f"  ✓ {caso}_FA: leído ({data.shape})")
        else:
            raise ValueError(f"Forma inesperada: {data.shape}")
    except Exception as e:
        datos_FA[caso] = generar_datos_FA_ejemplo(caso, seed=hash(caso)%100)
        print(f"  ~ {caso}_FA: usando datos de ejemplo")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
all_fa_vals = np.concatenate([d.flatten()[d.flatten()>0] for d in datos_FA.values()])
vmin_fa, vmax_fa = 0, all_fa_vals.max()

for ax, (caso, data) in zip(axes, datos_FA.items()):
    im = ax.imshow(data, cmap='hot_r', aspect='equal',
                   vmin=vmin_fa, vmax=vmax_fa, origin='upper')
    ax.set_title(f'Distribución Pin-by-Pin — {caso}', fontweight='bold')
    ax.set_xlabel('Columna')
    ax.set_ylabel('Fila')
    ax.set_xticks(range(0, 17, 4))
    ax.set_yticks(range(0, 17, 4))
    plt.colorbar(im, ax=ax, label='Potencia relativa', shrink=0.8)

plt.suptitle('Distribución radial pin-by-pin en el FA — NuScale',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Distribución axial de potencia

La distribución axial describe cómo varía la potencia a lo largo del eje del reactor (de abajo a arriba). Para un reactor sin barras de control insertadas, el perfil es aproximadamente **cosenoidal**:

$$P(z) = P_0 \cos\left(\frac{\pi z}{H_{ext}}\right)$$

donde $H_{ext}$ es la altura extrapolada del núcleo.

En la práctica, la presencia de barras de control, venenos combustibles y la densidad variable del refrigerante distorsionan este perfil.


In [ ]:
def generar_perfil_axial_ejemplo(caso='REF', n_nodos=26):
    """
    Genera un perfil axial sintético.
    La altura del núcleo NuScale activo es ~200 cm.
    """
    H = 200.0  # cm
    H_ext = H * 1.15  # altura extrapolada
    z = np.linspace(H/(2*n_nodos), H - H/(2*n_nodos), n_nodos)
    
    if caso == 'REF':
        # Perfil cosenoidal con ligera distorsión por barras
        p = np.cos(np.pi * (z - H/2) / H_ext)
        # Barras de control en la mitad superior reducen potencia
        p[n_nodos//2:] *= np.linspace(1.0, 0.85, n_nodos//2)
    elif caso == 'SIM':
        p = np.cos(np.pi * (z - H/2) / H_ext)
        p *= np.linspace(0.95, 1.05, n_nodos)  # ligera asimetría
    else:  # VAC
        # Void aumenta la importancia de la región inferior
        p = np.cos(np.pi * (z - H/2) / H_ext)
        p[:n_nodos//2] *= 1.1
        p[n_nodos//2:] *= 0.9
    
    p = np.maximum(p, 0)
    p = p / p.mean()
    return z, p


# Cargar o generar perfiles axiales
perfiles_axiales = {}
for caso in ['REF', 'SIM', 'VAC']:
    try:
        archivo_ax = P1 / f'{caso}_CORE.txt'
        if not archivo_ax.exists():
            raise FileNotFoundError()
        z, p = leer_distribucion_axial(archivo_ax)
        perfiles_axiales[caso] = (z, p)
        print(f"  ✓ {caso}: leído ({len(z)} nodos axiales)")
    except:
        z, p = generar_perfil_axial_ejemplo(caso)
        perfiles_axiales[caso] = (z, p)
        print(f"  ~ {caso}: usando datos de ejemplo ({len(z)} nodos)")

In [ ]:
# Gráfico comparativo de perfiles axiales
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colores = {'REF': 'steelblue', 'SIM': 'darkorange', 'VAC': 'forestgreen'}
estilos = {'REF': '-', 'SIM': '--', 'VAC': '-.'}

# Izquierda: todos en un gráfico
ax = axes[0]
for caso, (z, p) in perfiles_axiales.items():
    ax.plot(p, z, color=colores[caso], ls=estilos[caso], label=f'Caso {caso}', linewidth=2.5)

# Perfil cosenoidal teórico
z_ref = np.linspace(0, 200, 200)
H_ext = 230.0
p_cos = np.cos(np.pi * (z_ref - 100) / H_ext)
p_cos = np.maximum(p_cos, 0)
p_cos = p_cos / p_cos.mean()
ax.plot(p_cos, z_ref, 'k:', linewidth=1.5, alpha=0.6, label='Teórico (coseno)')

ax.set_xlabel('Potencia relativa [-]', fontsize=12)
ax.set_ylabel('Altura z [cm]', fontsize=12)
ax.set_title('Distribución Axial de Potencia\n(núcleo completo)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 200)
ax.axhline(100, color='gray', ls=':', alpha=0.5, label='Plano medio')

# Derecha: diferencia respecto al caso REF
ax2 = axes[1]
z_ref_arr, p_ref_arr = perfiles_axiales['REF']
for caso, (z, p) in perfiles_axiales.items():
    if caso == 'REF':
        continue
    # Interpolar si necesario
    p_interp = np.interp(z_ref_arr, z, p)
    diff = (p_interp - p_ref_arr) / p_ref_arr * 100
    ax2.plot(diff, z_ref_arr, color=colores[caso], ls=estilos[caso],
             label=f'{caso} - REF', linewidth=2.5)

ax2.axvline(0, color='gray', ls='--', alpha=0.7)
ax2.set_xlabel('Diferencia respecto a REF [%]', fontsize=12)
ax2.set_ylabel('Altura z [cm]', fontsize=12)
ax2.set_title('Diferencia relativa de potencia axial\n(% respecto a REF)', fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim(0, 200)

plt.tight_layout()
plt.show()

---
## 4. Factores de forma y factores de ingeniería

### 4.1 Factor de forma axial ($F_z$)

El **factor de forma axial** (o factor de pico axial) se define como:

$$F_z = \frac{P_{max}(z)}{\bar{P}} $$

donde $P_{max}(z)$ es la potencia máxima en cualquier nodo axial y $\bar{P}$ es la potencia media. Cuanto más cerca de 1.0, más plano es el perfil.

### 4.2 Factor de pico global ($F_q$)

El factor de pico global combina la información radial y axial:

$$F_q = F_{xy} \cdot F_z$$

Este factor limita la potencia máxima de la varilla de combustible más cargada y es el parámetro de seguridad más importante en el diseño del núcleo.


In [ ]:
# Calcular factores para cada caso
print("\nFactores de forma y pico de potencia")
print("=" * 60)
print(f"{'Caso':<8} {'F_xy':>8} {'F_z':>8} {'F_q = F_xy·F_z':>16} {'Interpretación'}")
print("-" * 60)

LIMITE_FQ = 2.50  # Límite típico para PWR

for caso in ['REF', 'SIM', 'VAC']:
    p_rad = datos_radiales[caso]
    _, p_ax = perfiles_axiales[caso]
    
    Fxy = p_rad.max() / p_rad.mean()
    Fz  = p_ax.max()  / p_ax.mean()
    Fq  = Fxy * Fz
    
    ok = "✓ OK" if Fq < LIMITE_FQ else "✗ EXCEDE LÍMITE"
    print(f"{caso:<8} {Fxy:>8.4f} {Fz:>8.4f} {Fq:>16.4f}   {ok}")

print("=" * 60)
print(f"\nLímite de diseño típico: F_q < {LIMITE_FQ}")

---
## 5. Análisis de sensibilidad

### 5.1 Sensibilidad a la variación de nivel de potencia

Estudiamos cómo cambia la distribución de potencia cuando el reactor opera a distintas fracciones de potencia nominal: 85%, 90%, 95% y 105%.

En un reactor PWR, el cambio de nivel de potencia afecta a la temperatura del moderador, que a su vez modifica las secciones eficaces (efecto Doppler y coeficiente de temperatura del moderador).


In [ ]:
def cargar_caso_sensibilidad_potencia(nivel_pct, datos_ref):
    """
    Carga o simula el caso de sensibilidad para un nivel de potencia dado.
    
    En el ejercicio real, cada nivel tiene su propio archivo COBAYA.
    Aquí simulamos el efecto: a más potencia, la distribución se aplana
    ligeramente por el efecto moderador (coeficiente negativo).
    """
    rng = np.random.default_rng(nivel_pct)
    factor = nivel_pct / 100.0
    # A mayor potencia: más temperatura → más absorción → perfil ligeramente más plano
    alfa_moderador = -0.03  # cm^-2/K, valor típico PWR
    delta_T = (factor - 1.0) * 25  # K de variación de temperatura
    efecto = 1 + alfa_moderador * delta_T * 0.01  # Corrección simplificada
    
    p_nuevo = datos_ref * efecto
    # El pico se atenúa un poco a mayor potencia
    p_nuevo = p_nuevo / p_nuevo.mean()
    return p_nuevo


niveles_potencia = [85, 90, 95, 100, 105]
datos_sens_potencia = {}
p_ref = datos_radiales['REF'].copy()

for nivel in niveles_potencia:
    datos_sens_potencia[nivel] = cargar_caso_sensibilidad_potencia(nivel, p_ref)

# Gráfico: evolución de F_xy con el nivel de potencia
Fxy_niveles = [datos_sens_potencia[n].max()/datos_sens_potencia[n].mean() for n in niveles_potencia]
keff_estimado = [1.0 + 0.003*(n-100)/100 for n in niveles_potencia]  # Estimación simplificada

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(niveles_potencia, Fxy_niveles, 'o-', color='steelblue', markersize=8, linewidth=2)
ax.axhline(Fxy_niveles[2], color='gray', ls='--', alpha=0.5, label='100% nominal')
ax.set_xlabel('Nivel de potencia [%]', fontsize=12)
ax.set_ylabel('Factor de pico radial F_xy [-]', fontsize=12)
ax.set_title('Sensibilidad de F_xy al nivel de potencia', fontweight='bold')
ax.legend()
for x, y in zip(niveles_potencia, Fxy_niveles):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(5, 5), fontsize=9)

ax2 = axes[1]
ax2.plot(niveles_potencia, keff_estimado, 's--', color='darkorange', markersize=8, linewidth=2)
ax2.axhline(1.0, color='red', ls=':', alpha=0.7, label='Criticidad (k=1)')
ax2.set_xlabel('Nivel de potencia [%]', fontsize=12)
ax2.set_ylabel('k_eff estimado [-]', fontsize=12)
ax2.set_title('Variación de k_eff con el nivel de potencia\n(efecto temperatura del moderador)', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

### 5.2 Sensibilidad a la temperatura del combustible (efecto Doppler)

El **efecto Doppler** es el ensanchamiento de las resonancias de absorción del U-238 con la temperatura del combustible. Al aumentar la temperatura del combustible, aumenta la absorción parasitaria y disminuye $k_{eff}$ — esto proporciona un mecanismo de **retroalimentación negativa** fundamental para la seguridad.

El coeficiente de temperatura del combustible (CTC) o coeficiente Doppler se define:

$$\alpha_D = \frac{d k_{eff}}{d T_f} \approx \frac{\Delta k_{eff}}{\Delta T_f}$$


In [ ]:
# Análisis de sensibilidad a la temperatura del combustible
# Rango típico: T_fuel = 520 K a 560 K (condiciones moderador)

temperaturas = np.array([520, 530, 540, 550, 560])  # K
T_ref = 540.0  # K temperatura de referencia

# Efecto Doppler: dk/dT ≈ -3×10⁻⁵ /K (valor típico para UO2 en PWR)
alfa_doppler = -3.0e-5  # /K
keff_ref = 1.0050  # caso REF a T_nominal

keff_vs_T = keff_ref + alfa_doppler * (temperaturas - T_ref)

# Calcular coeficiente numérico
dk_dT = np.gradient(keff_vs_T, temperaturas)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(temperaturas, keff_vs_T * 1e5, 'o-', color='crimson', markersize=8, linewidth=2.5)
ax.axhline(1.0 * 1e5, color='gray', ls='--', alpha=0.5, label='k=1 (criticidad)')
ax.set_xlabel('Temperatura del combustible T_f [K]', fontsize=12)
ax.set_ylabel('k_eff × 10⁵ [pcm]', fontsize=12)
ax.set_title('Efecto Doppler: k_eff vs T_combustible', fontweight='bold')
ax.legend()
for T, k in zip(temperaturas, keff_vs_T):
    ax.annotate(f'{(k-keff_ref)*1e5:+.1f} pcm', (T, k*1e5),
                textcoords='offset points', xytext=(5, 5), fontsize=9)

ax2 = axes[1]
ax2.plot(temperaturas, dk_dT * 1e5, 's-', color='purple', markersize=8, linewidth=2.5)
ax2.axhline(0, color='gray', ls='--', alpha=0.5)
ax2.set_xlabel('Temperatura del combustible T_f [K]', fontsize=12)
ax2.set_ylabel('dk_eff/dT [pcm/K]', fontsize=12)
ax2.set_title('Coeficiente Doppler (dk/dT_f)\n[debe ser negativo para estabilidad]', fontweight='bold')
ax2.fill_between(temperaturas, dk_dT*1e5, 0,
                  where=dk_dT<0, alpha=0.2, color='green', label='Retroalimentación negativa ✓')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nCoeficiente Doppler promedio: {alfa_doppler*1e5:.2f} pcm/K")
print(f"Variación total en {temperaturas.min()}-{temperaturas.max()} K: ",
      f"{(keff_vs_T[-1]-keff_vs_T[0])*1e5:.1f} pcm")

---
## 6. Resumen y conclusiones


In [ ]:
print("=" * 60)
print("RESUMEN — Neutrónica con COBAYA")
print("=" * 60)

print("\n1. DISTRIBUCIÓN RADIAL")
for caso in ['REF', 'SIM', 'VAC']:
    p = datos_radiales[caso]
    print(f"   {caso}: F_xy = {p.max()/p.mean():.4f} | "
          f"σ/μ = {p.std()/p.mean()*100:.1f}%")

print("\n2. DISTRIBUCIÓN AXIAL")
for caso in ['REF', 'SIM', 'VAC']:
    _, p = perfiles_axiales[caso]
    print(f"   {caso}: F_z = {p.max()/p.mean():.4f} | "
          f"Máximo a z ≈ {perfiles_axiales[caso][0][np.argmax(p)]:.0f} cm")

print("\n3. COEFICIENTES DE SEGURIDAD")
print(f"   Coeficiente Doppler: {alfa_doppler*1e5:.2f} pcm/K (negativo = estable)")

print("\n→ Siguiente: 02_termohidraulica_ctf.ipynb")
print("=" * 60)